In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [3]:
DATASET_PATH = "/kaggle/input/datasets/sugamg/wildlife-aerial-imagery-dataset/WAID-final - Copy"

In [4]:
import os
import cv2
from tqdm import tqdm

INPUT_IMG = DATASET_PATH + "/images"
INPUT_LBL = DATASET_PATH + "/labels"

OUTPUT_PATH = "/kaggle/working/processed_dataset"

# Create folders
for split in ["train", "test"]:
    os.makedirs(f"{OUTPUT_PATH}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/labels/{split}", exist_ok=True)

def process_images(split):
    in_path = f"{INPUT_IMG}/{split}"
    out_path = f"{OUTPUT_PATH}/images/{split}"

    for img_name in tqdm(os.listdir(in_path)):
        img_path = os.path.join(in_path, img_name)
        img = cv2.imread(img_path, -1)

        if img is None:
            continue

        # 🔥 Handle 16-bit thermal
        if img.dtype == 'uint16':
            img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
            img = img.astype('uint8')

        # 🔥 Handle grayscale
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        cv2.imwrite(os.path.join(out_path, img_name), img)

def copy_labels(split):
    in_lbl = f"{INPUT_LBL}/{split}"
    out_lbl = f"{OUTPUT_PATH}/labels/{split}"

    for file in os.listdir(in_lbl):
        src = os.path.join(in_lbl, file)
        dst = os.path.join(out_lbl, file)
        with open(src, "r") as f:
            lines = f.readlines()

        # (optional sanity check)
        clean_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                clean_lines.append(line)

        with open(dst, "w") as f:
            f.writelines(clean_lines)

# Run preprocessing
for split in ["train", "test"]:
    process_images(split)
    copy_labels(split)

print("Preprocessing Done!")

100%|██████████| 402/402 [00:05<00:00, 72.95it/s]


Preprocessing Done!


In [5]:
data_yaml = f"""
path: {OUTPUT_PATH}

train: images/train
val: images/test

nc: 10
names:
  0: sheep
  1: cattle
  2: seal
  3: camelus
  4: kiang
  5: zebra
  6: crocodile
  7: elephant
  8: deer
  9: horse
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created")

data.yaml created


In [6]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=800,     
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=Fa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a8b1839bbf0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.0

In [7]:
!ls runs/detect/train

args.yaml			 results.csv	      val_batch0_labels.jpg
BoxF1_curve.png			 results.png	      val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg     val_batch1_pred.jpg
BoxR_curve.png			 train_batch2.jpg     val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch4920.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch4921.jpg  weights
labels.jpg			 train_batch4922.jpg


In [8]:
model.predict(
    source=f"{OUTPUT_PATH}/images/test",
    save=True,
    conf=0.25
)


image 1/402 /kaggle/working/processed_dataset/images/test/dataset_2500.jpg: 800x800 23 elephants, 6.0ms
image 2/402 /kaggle/working/processed_dataset/images/test/dataset_2501.jpg: 800x800 9 cattles, 6.2ms
image 3/402 /kaggle/working/processed_dataset/images/test/dataset_2502.JPG: 640x800 1 deer, 44.4ms
image 4/402 /kaggle/working/processed_dataset/images/test/dataset_2503.JPG: 640x800 3 deers, 6.8ms
image 5/402 /kaggle/working/processed_dataset/images/test/dataset_2504.JPG: 640x800 1 deer, 6.4ms
image 6/402 /kaggle/working/processed_dataset/images/test/dataset_2505.JPG: 640x800 3 deers, 6.2ms
image 7/402 /kaggle/working/processed_dataset/images/test/dataset_2506.JPG: 640x800 (no detections), 5.9ms
image 8/402 /kaggle/working/processed_dataset/images/test/dataset_2507.JPG: 640x800 1 deer, 6.2ms
image 9/402 /kaggle/working/processed_dataset/images/test/dataset_2508.JPG: 640x800 1 deer, 6.1ms
image 10/402 /kaggle/working/processed_dataset/images/test/dataset_2509.JPG: 640x800 1 deer, 6.3

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'sheep', 1: 'cattle', 2: 'seal', 3: 'camelus', 4: 'kiang', 5: 'zebra', 6: 'crocodile', 7: 'elephant', 8: 'deer', 9: 'horse'}
 obb: None
 orig_img: array([[[21, 25, 20],
         [13, 17, 12],
         [37, 43, 38],
         ...,
         [41, 78, 58],
         [24, 59, 39],
         [19, 51, 32]],
 
        [[16, 20, 15],
         [ 2,  8,  3],
         [23, 29, 24],
         ...,
         [40, 77, 57],
         [23, 58, 38],
         [18, 50, 31]],
 
        [[13, 19, 14],
         [ 3, 10,  5],
         [17, 27, 21],
         ...,
         [52, 89, 69],
         [44, 79, 59],
         [43, 75, 56]],
 
        ...,
 
        [[19, 39, 27],
         [50, 70, 58],
         [64, 84, 72],
         ...,
         [32, 36, 30],
         [25, 32, 25],
         [23, 30, 23]],
 
        [[23, 40, 29],
         [50, 67, 56],
         [54, 71, 60]